# Attribute 1 - Threatened Environment
#### Rationale
Increase restoration on land environments that are the most under-represented (there will be some overlap with PNV representation but TEC is  based on areas of similar abiotic features, and is a recognised system to prioritise protection and restoration (NZ Govt priorities, 2007), vs the catchment-based representation of the PNVs


#### Data layers
Threatened Environments Classification 2012 From: https://lris.scinfo.org.nz/layer/48288-threatened-environments-classification-2012/
Note latest version is 2012, we could update using LCDB 2018 – or check with MWLR if they plan to update it


## Scoring
- 1.0: TEC1
- 0.8: TEC2
- 0.6: TEC3
- 0.4: TEC4
- 0.2: TEC5
- 0.0: TEC6


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import json
import math
import os
import time
from os import listdir

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import pyproj
import rasterio
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.transform import from_origin
from shapely.geometry import Point, Polygon, box

from constants import small_polygon_threshold, m2_to_ha, x_resolution, y_resolution, keep_cols, keep_cols_catch

## Load layers

In [4]:
%%time
# area of interest
aoi = gpd.read_file("../BaseLayersEco-index/Eco-index_Catchments_v080623.gpkg")
aoi.sindex
aoi.shape

restorable = gpd.read_file('../BaseLayersEco-index/Eco-index_RestorableAreas_v290824.gpkg')
restorable.sindex

aoi = aoi.overlay(restorable)
aoi.to_file('../BaseLayersEco-index/Eco-index_RestorableAreas__Catchments_v290824.gpkg')

C:\Users\dav\miniconda3_9\envs\eco\Lib\site-packages\geopandas\geodataframe.py:2675: UserWarning: `keep_geom_type=True` in overlay resulted in 127 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  return geopandas.overlay(


CPU times: total: 2min 34s
Wall time: 2min 34s


In [5]:
%%time
# C:\Users\dav\Documents\EcoIndex\_GIS_CORE\layers-external\lris-threatened-environments-classification-2012-SHP
threat_path = r"..\BaseLayersExternal\lris-threatened-environments-classification-2012-SHP\threatened-environments-classification-2012.shp"
threat = gpd.read_file(threat_path)
threat.sindex
threat.shape

CPU times: total: 13.7 s
Wall time: 13.9 s


(815143, 13)

In [6]:
tec_class_pixel_mapping = {
    # 0 : ??,
    1: 10,
    2: 8,
    3: 6,
    4: 4,
    5: 2,
    6: 0,
}
threat["PixelScore"] = threat.TEC12_Clas.map(tec_class_pixel_mapping)
threat["PixelDesc"] = threat["TEC12_Cl_1"].copy()
threat = threat[~threat["PixelScore"].isna()].copy()
threat["PrioOption"] = "Threatened Environment"

In [7]:
import time

gdfs = []
for n_catch, catchment in enumerate(aoi.Catchment.sort_values().unique()):
    current_time = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(time.time()))
    print(f"\n{n_catch}_{catchment.upper()}      {current_time}")
    start_time = time.time()

    sub_aoi = aoi[aoi.Catchment == catchment].copy()
    sub_aoi.sindex

    threat_env = threat.overlay(sub_aoi, how="intersection")

    if threat_env.shape[0] > 0:
        threat_env["Area_ha"] = threat_env.area * m2_to_ha
        threat_env = threat_env[~pd.isna(threat_env.PixelScore)]
        threat_env = (
            threat_env[threat_env.Area_ha > small_polygon_threshold]
            .copy()
            .reset_index(drop=True)
        )
        threat_env["Area_ha"] = threat_env["Area_ha"].round(2)
        threat_env["Catchment"] = catchment
    
        threat_env.to_file(
            f"../OutputArtifacts/A01_ThreatenedEnvironment/A01_Catchments/{str(n_catch).zfill(3)}_{catchment}_threat_env_20240829.gpkg"
        )
        
    time_diff = time.time() - start_time
    formatted_time_diff = time.strftime("%M:%S", time.gmtime(time_diff))
    print(f"Threat env saved. Elapsed time: {formatted_time_diff}")
    gdfs.append(threat_env)
    



0_APARIMA      2024-09-03 21:58:07
Threat env saved. Elapsed time: 00:23

1_ASHBURTON-HINDS      2024-09-03 21:58:31
Threat env saved. Elapsed time: 00:16

2_AUCKLAND BASIN      2024-09-03 21:58:47
Threat env saved. Elapsed time: 00:26

3_AUCKLAND OFFSHORE ISLANDS      2024-09-03 21:59:13
Threat env saved. Elapsed time: 00:08

4_AWATERE      2024-09-03 21:59:22
Threat env saved. Elapsed time: 00:28

5_BANKS PENINSULA      2024-09-03 21:59:50
Threat env saved. Elapsed time: 00:19

6_BAY OF PLENTY OFFSHORE ISLANDS      2024-09-03 22:00:09
Threat env saved. Elapsed time: 00:08

7_BLUFF OFFSHORE      2024-09-03 22:00:18
Threat env saved. Elapsed time: 00:08

8_BULLER      2024-09-03 22:00:27
Threat env saved. Elapsed time: 00:10

9_CATLINS      2024-09-03 22:00:37
Threat env saved. Elapsed time: 00:17

10_CLARENCE      2024-09-03 22:00:55
Threat env saved. Elapsed time: 00:29

11_CLUTHA      2024-09-03 22:01:24
Threat env saved. Elapsed time: 13:10

12_CONWAY      2024-09-03 22:14:34
Thre

In [9]:
concatenated_gdf = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True))
# concatenated_gdf.sindex
concatenated_gdf.drop_duplicates()[keep_cols].to_file(
    "../OutputArtifacts/A01_ThreatenedEnvironment/A01_Threatened_Environment_20240624.gpkg"
)

In [8]:
## Don't want to Rasterize the whole country
## ... or do we?


# %%time
# from gis_analysis_functions import rasterize_and_save
# rasterize_and_save(
#     aoi,
#     concatenated_gdf,
#     10,
#     10,
#     "output_layers/attr01/01_Threatened_Environment_10m_20240624.tif",
# )

In [10]:
concatenated_gdf.head()

NameError: name 'concatenated_gdf' is not defined